# 🚀 Huấn Luyện PhoBERT-BiLSTM-CRF Trên Google Colab (GPU Tesla T4)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barackvn/NLP/blob/main/notebooks/02_train_colab_gpu_t4.ipynb)

**Đề tài:** Nhận diện chuỗi ngôn ngữ xúc phạm tiếng Việt (ViHOS)  
**Người phụ trách:** Hoàng Võ Minh Tuấn (Trainer - 26410146)  
**Môi trường:** GPU Tesla T4 (16GB VRAM), PyTorch, HuggingFace Transformers, PyTorch-CRF, Seqeval  
**GitHub Repo chính thức:** [https://github.com/barackvn/NLP](https://github.com/barackvn/NLP)


## Bước 1: Kiểm tra cấu hình GPU Tesla T4
Đảm bảo bạn đã chọn **Runtime -> Change runtime type -> T4 GPU** trước khi chạy!


In [ ]:
# Kiểm tra GPU Colab
!nvidia-smi


## Bước 2: Tải Mã Nguồn & Dữ Liệu từ GitHub Chính Thức
Toàn bộ code đã sửa lỗi CRF và dữ liệu ViHOS đầy đủ (11.056 câu) sẽ được nạp tự động.


In [ ]:
import os
%cd /content
if os.path.exists('/content/NLP'):
    !rm -rf /content/NLP

# Clone repository chính thức
!git clone https://github.com/barackvn/NLP.git
%cd /content/NLP
!pwd


## Bước 3: Cài đặt các thư viện cần thiết


In [ ]:
!pip install -q transformers accelerate


## Bước 4: Kết nối Google Drive để lưu checkpoint vĩnh viễn


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/ViHOS_Checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"✅ Thư mục sao lưu Drive: {CHECKPOINT_DIR}")


## Bước 5: Huấn Luyện Mô Hình Đề Xuất (PhoBERT-BiLSTM-CRF)
- **Cấu hình:** Epochs = 5, Batch size = 16, PhoBERT LR = `2e-5`, BiLSTM-CRF LR = `1e-3`
- **Cơ chế:** Word-level Pooling + Contiguous Viterbi Decoding (Đã sửa triệt để lỗi CRF)
- **Early Stopping:** patience = 3 trên tập DEV


In [ ]:
# Cài đặt tự động để đảm bảo thư viện luôn sẵn sàng kể cả khi chưa chạy Bước 3
!pip install -q transformers accelerate

!python -m src.train \
    --model_type phobert_bilstm_crf \
    --train_path data/processed/train.json \
    --dev_path data/processed/dev.json \
    --save_path {CHECKPOINT_DIR}/best_phobert_bilstm_crf.pt \
    --epochs 5 \
    --batch_size 16 \
    --lr_phobert 2e-5 \
    --lr_head 1e-3 \
    --max_length 128 \
    --patience 3


## Bước 6: Huấn Luyện 2 Mô Hình Đối Chứng (Ablation Baselines)
1. **PhoBERT-Linear (Baseline):** Softmax độc lập từng token.
2. **PhoBERT-CRF (Bóc tách Ablation):** Đánh giá vai trò của tầng BiLSTM.


In [ ]:
# 1. Baseline: PhoBERT-Linear
!python -m src.train \
    --model_type phobert_linear \
    --train_path data/processed/train.json \
    --dev_path data/processed/dev.json \
    --save_path {CHECKPOINT_DIR}/baseline_phobert_linear.pt \
    --epochs 5 \
    --batch_size 16 \
    --lr_phobert 2e-5 \
    --lr_head 1e-3

# 2. Baseline bóc tách: PhoBERT-CRF
!python -m src.train \
    --model_type phobert_crf \
    --train_path data/processed/train.json \
    --dev_path data/processed/dev.json \
    --save_path {CHECKPOINT_DIR}/baseline_phobert_crf.pt \
    --epochs 5 \
    --batch_size 16 \
    --lr_phobert 2e-5 \
    --lr_head 1e-3


## Bước 7: Đánh Giá So Sánh Cả 3 Mô Hình Trên Tập Test ViHOS (1.106 câu)
Đo lường Span-Precision, Span-Recall, Span-F1 chuẩn mực bằng seqeval.


In [ ]:
print('='*70)
print('1. KẾT QUẢ TEST: PhoBERT-Linear (Baseline):')
!python -m src.evaluate --model_type phobert_linear --checkpoint {CHECKPOINT_DIR}/baseline_phobert_linear.pt --test_path data/processed/test.json

print('='*70)
print('2. KẾT QUẢ TEST: PhoBERT-CRF (Bóc tách Ablation):')
!python -m src.evaluate --model_type phobert_crf --checkpoint {CHECKPOINT_DIR}/baseline_phobert_crf.pt --test_path data/processed/test.json

print('='*70)
print('3. KẾT QUẢ TEST: PhoBERT-BiLSTM-CRF (Đề xuất SOTA):')
!python -m src.evaluate --model_type phobert_bilstm_crf --checkpoint {CHECKPOINT_DIR}/best_phobert_bilstm_crf.pt --test_path data/processed/test.json


## Bước 8: Tải Checkpoint Về Máy Cá Nhân Để Chạy Web App Demo


In [ ]:
from google.colab import files
# Tải checkpoint tốt nhất về máy qua trình duyệt
files.download(f"{CHECKPOINT_DIR}/best_phobert_bilstm_crf.pt")
